# Unit Testing Exercises

Unit tests help ensure your code works as expected and make your RAP pipeline more robust and reproducible. In this exercise, you'll learn how to write and run unit tests using pytest.

## Why are unit tests important?

Unit tests check that individual functions behave as expected. They:
- Help catch bugs early
- Make code easier to maintain
- Support reproducibility and automation
- Give confidence when refactoring or adding new features

Read more in the [pytest documentation](https://docs.pytest.org/en/stable/).

## Exercise 1: Review and Adapt an existing unit test

The function `clean_health_data` in `src/python_rap_demo/cleaning.py` has an existing unit test in
`tests/test_cleaning.py`.

**Task:** 
1. Run the existing unit test for `clean_health_data` to understand how it works. 
2. Modify the function `clean_health_data` and re-run the tests to understand what causes the tests to pass or fail, then modify the unit test so that the tests passes again.

There is an example of a modified function below however it does not have to be used, feel free to modify the function and experiment with the tests without following the set exercise.

In [ ]:
def clean_health_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean health data by dropping rows with missing values in key columns.

    Args:
        df (pd.DataFrame): Raw health data.

    Returns:
        pd.DataFrame: Cleaned health data with no missing values in critical columns.
    """
    df = df.copy()

    # Drop rows with missing values in height_cm, weight_kg, or diagnosis columns
    df = df.dropna(subset=["height_cm", "weight_kg", "diagnosis"])

    # Fill missing smoker values with 'Yes'
    df["smoker"] = df["smoker"].fillna("Yes")

    # Ensure gender is uppercase
    df["gender"] = df["gender"].str.upper()

    return df


## Exercise 2: Write a simple unit test for a new function

Suppose you have created a function called `impute_by_group` in `src/python_rap_demo/cleaning.py`.

**Task:**
1. Create a new test file in the `tests/` folder (e.g., `test_cleaning.py`).
2. Write a unit test for `impute_by_group` that checks it correctly imputes missing values.
3. Use pytest for your test.

**Hints:**
- Read on for more guidance

## Exercise 2a: Walkthrough - Write a unit test for flag_missing

Let's start with a simple function called `flag_missing`. This function adds a new column to your DataFrame to flag missing values in specified columns.

Below is the code for `flag_missing`:


In [ ]:
from typing import List

import pandas as pd


def flag_missing(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """
    Add boolean columns to flag missing values for specified columns.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (List[str]): List of columns to flag.

    Returns:
        pd.DataFrame: DataFrame with new _imputed columns.
    """
    df = df.copy()
    for col in columns:
        df[f"{col}_imputed"] = df[col].isna()
    return df


# Example usage:
df = pd.DataFrame({"height_cm": [170, None, 160], "weight_kg": [70, 80, None]})
flagged = flag_missing(df, ["height_cm", "weight_kg"])
print(flagged)

### How to write a unit test for `flag_missing`

1. Add the `flag_missing` function to the `src/python_repo_demo/cleaning.py` file if you have not done so already.
2. Open the `test_cleaning.py` file in the `tests/` folder.
3. Import the `flag_missing` function from your cleaning module at the top of your test file:
   ```python
   from python_rap_demo.cleaning import flag_missing
   ```
4. Import pandas for creating test data (if not already there):
   ```python
   import pandas as pd
   ```
5. Write a test function using the `def` keyword. The function name should start with `test_`:
   ```python
   def test_flag_missing():
       # Create a small DataFrame with missing values
       df = pd.DataFrame({"height_cm": [170, None], "weight_kg": [70, None]})
       # Call the function
       flagged = flag_missing(df, ["height_cm", "weight_kg"])
       # Check the results using assert statements
       assert flagged["height_cm_imputed"].tolist() == [False, True]
       assert flagged["weight_kg_imputed"].tolist() == [False, True]
   ```
6. Save the file and run your tests in the terminal using the command:
   ```
   pytest tests
   ```
7. If the test passes, your function works as expected. If it fails, check the error message and review your code.



#### Understanding the test_flag_missing function

Let's break down how the `test_flag_missing` function works:

- **Function name:** The name starts with `test_` so pytest will automatically find and run it.
- **Creating test data:** We use pandas to create a small DataFrame `df` with some missing values. This lets us check if our function works as expected.
- **Calling the function:** We call `flag_missing` and pass in our test DataFrame and the columns we want to check for missing values.
- **Checking the results:** We use `assert` statements to check that the new columns (e.g., `height_cm_imputed`, `weight_kg_imputed`) correctly show `True` for missing values and `False` otherwise. If the output matches what we expect, the test will pass.
- **Why use assert?** Assert statements are a way to say "this must be true". If the condition is not true, pytest will show an error and the test will fail.

This approach helps you check that your function does what you expect, using simple, readable code. You can add more assert statements or test with different data to make your tests even stronger.


## Exercise 2b: Write a unit test for impute_by_group

Now try writing your own tests for the following function. Use the walkthrough above as a guide.

**Hints:**
- Import the function at the top of your test file.
- Use a small pandas DataFrame with missing values for testing.
- Use `assert` statements to check the output.

Below is the function definition:


In [ ]:
def impute_by_group(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Impute missing values in a column using group means, fallback to overall mean if needed.
    Rounds imputed values to 1 decimal place for consistency.

    Args:
        df (pd.DataFrame): Input DataFrame.
        col (str): Column to impute.
        group_col (str): Column to group by.

    Returns:
        pd.Series: Imputed column.
    """
    group_means = df.groupby(group_col)[col].transform("mean")
    result = df[col].fillna(group_means)
    result = result.fillna(df[col].mean())
    return result.round(1)

**Task:**
- Add the `flag_missing`, `impute_by_group` functions to the `src/python_rap_demo/cleaning.py` file if you have not done so already.
- Open the `test_cleaning.py` file in the `tests/` folder.
- Write a unit test for `impute_by_group` that checks missing values are correctly imputed using group means.
- Use pytest and assert statements in your test file.

**Note:**
The tests will run for these functions, but they will not affect the output unless you implement them in the pipeline. See `02_modules_solutions` for a guide on how to do this


## Exercise 3: Run your unit tests

**Task:**
- Run all tests in the `tests/` folder using the command below:
  ```cmd
  pytest tests
  ```
- Check the output to see if your tests pass or fail.

**Note:**
The tests will run, but they will not affect the outputs unless you add them into the pipeline.

## Exercise 4: Stretch - Check test coverage

Test coverage shows how much of your code is tested by unit tests.

**Task:**
- Install coverage if needed:
  ```cmd
  pip install coverage
  ```
- Run:
  ```cmd
  coverage run -m pytest tests
  coverage report
  ```
- Check which lines of your code are covered by tests.

## Exercise 5: Stretch - Try parameterisation in pytest

Parameterisation lets you run the same test with different inputs.

**Task:**
- Use `@pytest.mark.parametrize` in your test file to test your function with multiple input cases.
- See the [pytest parameterisation docs](https://docs.pytest.org/en/stable/example/parametrize.html) for examples.